# 04. Casos Curados de Tesis (`thesis_curated`) y Cálculo de Ampacidad

Trabaja con los casos de cables subterráneos multicapa (`xlpe_discrete_layers`, `xlpe_backfill`) e
ilustra la inicialización sin malla `rbf_readout`, el muestreo focalizado y la bisección térmica
de ampacidad mediante `AmpacitySolver`.

In [ ]:
import sys
from pathlib import Path

# Localiza PINN_V4 automáticamente aunque se ejecute desde dentro de notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (SRC, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("Raíz PINN_V4:", ROOT)

In [ ]:
import pinn_thermal.models
from pinn_thermal.core.ampacity import AmpacitySolver
from pinn_thermal.core.trainer import PINNTrainer, TrainConfig
from pinn_thermal.visualization import GridSampler
from frontend.state import WorkspaceStore

store = WorkspaceStore(ROOT / "frontend" / "workspace", ROOT / "cases")
prob = store.load_problem("xlpe_discrete_layers")
print("Regiones de xlpe_discrete_layers:", [(r.name, r.kind, r.priority) for r in prob.regions])

cfg = TrainConfig(
    adam_steps=200,
    lbfgs_steps=15,
    width=24,
    depth=3,
    focused=True,
    resample_every=100,
    initializer="rbf_readout",
)
art = PINNTrainer(cfg).train(prob, "M1_EXACT", store.runs_dir / "nb04_xlpe_M1_EXACT", overwrite=True)
print("Evaluación M1_EXACT:", art.result["evaluation"])

# Corte vertical por x = 0 atravesando el cable y la interfaz de estratos y = -1.0 m
cut = GridSampler.sample_line_cut({"M1_EXACT": art}, prob, (0.0, -3.5), (0.0, 0.0), n_points=120)
print("T max sobre corte vertical:", float(max(cut["series"]["M1_EXACT"])), "°C")

# Cálculo rápido de ampacidad usando inicialización RBF + L-BFGS determinista
amp_cfg = TrainConfig(
    adam_steps=0,
    lbfgs_steps=5,
    lbfgs_iterations=5,
    width=20,
    depth=2,
    device="cpu",
    initializer="rbf_readout",
    initializer_settings={"n_per_region": 120, "stencil_size": 16},
)
amp_res = AmpacitySolver(
    R20_ohm_m=2.0e-4,
    target_temp_C=90.0,
    relaxation=0.8,
    max_iters=2,
    coupling_tol_K=1.0,
    source_tol_rel=0.05,
    max_coupling_iters=25,
).solve_with_pinn(
    prob, architecture="M1", config=amp_cfg, work_dir=store.runs_dir / "nb04_ampacity_tmp", initial_upper_A=700.0
)
print(
    f"Corriente I_mid={amp_res.I_mid_A:.2f} A | potencia={amp_res.power_W_m:.2f} W/m | "
    f"Tmax={amp_res.Tmax_at_mid_C:.2f} °C | convergió={amp_res.converged}"
)